In [0]:

# ================================================
# ETAPA 18 - VALIDAÇÃO DO MODELO COM INTERAÇÕES
# TESTE COMPLETO, SEM NOVO TREINAMENTO
# ================================================

from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

# 1. Aplicar o modelo piloto ao teste completo
previsoes_v2_completas = modelo_interacoes.transform(
    teste_v2.select(
        "usuario_id",
        "destino_id",
        "destino",
        F.col("target").cast("double").alias("label"),
        F.col("features_com_interacoes").alias("features")
    )
)

# 2. Calcular ROC-AUC
avaliador_v2 = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc_v2_completo = avaliador_v2.evaluate(
    previsoes_v2_completas
)

print("ROC-AUC no teste completo:", round(auc_v2_completo, 4))

# 3. Extrair probabilidade de aceitação
previsoes_v2_completas = (
    previsoes_v2_completas
    .withColumn(
        "prob_sim",
        vector_to_array("probability")[1]
    )
)

# 4. Verificar se há variação entre destinos
# para o mesmo viajante
diagnostico_v2 = (
    previsoes_v2_completas
    .groupBy("usuario_id")
    .agg(
        F.count("*").alias("destinos_avaliados"),
        F.countDistinct(
            F.round("prob_sim", 6)
        ).alias("probabilidades_distintas"),
        F.min("prob_sim").alias("prob_min"),
        F.max("prob_sim").alias("prob_max")
    )
    .filter(F.col("destinos_avaliados") >= 5)
)

# 5. Resumo da diferenciação de destinos
display(
    diagnostico_v2.agg(
        F.count("*").alias("viajantes_com_5_mais_destinos"),
        F.sum(
            F.when(
                F.col("probabilidades_distintas") > 1, 1
            ).otherwise(0)
        ).alias("viajantes_com_diferenciacao")
    )
)

print("Validação concluída!")


In [0]:

# ================================================
# ETAPA 17 - MODELO COM INTERAÇÕES
# TREINAMENTO PILOTO STRAVL
# ================================================

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F

# 1. Reutilizar os vetores da Etapa 9
treino_v2 = assembler_interacoes.transform(
    treino_interacoes_completo
)

teste_v2 = assembler_interacoes.transform(
    teste_interacoes_completo
)

# 2. Selecionar somente colunas necessárias
treino_piloto = (
    treino_v2
    .select(
        F.col("features_com_interacoes").alias("features"),
        F.col("target").cast("double").alias("label")
    )
    .sample(False, 0.05, seed=42)
)

teste_piloto = (
    teste_v2
    .select(
        F.col("features_com_interacoes").alias("features"),
        F.col("target").cast("double").alias("label")
    )
    .sample(False, 0.05, seed=43)
)

print("Treino piloto:", treino_piloto.count())
print("Teste piloto:", teste_piloto.count())

# 3. Configurar modelo
lr_interacoes = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    regParam=0.5,
    elasticNetParam=0.0,
    standardization=False
)

print("Treinando modelo com interações...")

modelo_interacoes = lr_interacoes.fit(treino_piloto)

print("Modelo com interações treinado!")

# 4. Avaliar
previsoes_interacoes = modelo_interacoes.transform(
    teste_piloto
)

avaliador = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc_interacoes = avaliador.evaluate(
    previsoes_interacoes
)

print(
    "ROC-AUC com interações:",
    round(auc_interacoes, 4)
)


In [0]:

# ================================================
# ETAPA 16 - DIAGNÓSTICO DE PERSONALIZAÇÃO
# ================================================

from pyspark.sql import functions as F

diagnostico = (
    resultado_personalizacao
    .groupBy("usuario_id")
    .agg(
        F.count("*").alias("destinos_avaliados"),
        F.countDistinct("prob_sim").alias(
            "probabilidades_diferentes"
        ),
        F.min("prob_sim").alias("prob_min"),
        F.max("prob_sim").alias("prob_max")
    )
    .withColumn(
        "amplitude",
        F.col("prob_max") - F.col("prob_min")
    )
)

display(diagnostico)

print("Diagnóstico concluído!")


In [0]:

# ================================================
# ETAPA 15 - TESTE DE PERSONALIZAÇÃO
# MESMOS DESTINOS PARA DIFERENTES VIAJANTES
# ================================================

from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import VectorAssembler

# 1. Selecionar 5 viajantes do teste
usuarios_amostra = (
    teste_features
    .select("usuario_id")
    .distinct()
    .orderBy("usuario_id")
    .limit(5)
)

# 2. Recuperar um perfil por viajante
perfis_amostra = (
    teste_features
    .join(usuarios_amostra, "usuario_id")
    .dropDuplicates(["usuario_id"])
    .select("usuario_id", *colunas_features)
)

# 3. Selecionar os mesmos 20 destinos
destinos_amostra = (
    destinos_indexados
    .orderBy("destino_id")
    .limit(20)
)

# 4. Combinar cada viajante com os 20 destinos
candidatos = perfis_amostra.crossJoin(
    destinos_amostra
)

# 5. Codificar os destinos com o encoder já treinado
candidatos = encoder_model.transform(candidatos)

# 6. Criar vetor de características
candidatos = assembler.transform(candidatos)

# 7. Aplicar Random Forest
resultado_personalizacao = (
    modelo_rf.transform(candidatos)
    .withColumn(
        "prob_sim",
        vector_to_array("probability")[1]
    )
)

# 8. Mostrar as probabilidades por viajante
display(
    resultado_personalizacao
    .select(
        "usuario_id",
        "destino_id",
        "prob_sim"
    )
    .orderBy("usuario_id", F.desc("prob_sim"))
)

print("Teste de personalização concluído!")


In [0]:

# ================================================
# ETAPA 14 - DIVERSIDADE DOS RANKINGS TOP 5
# ================================================

from pyspark.sql import functions as F

# 1. Montar assinatura ordenada do Top 5
rankings_usuarios = (
    top5_rf
    .groupBy("usuario_id")
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct("posicao", "destino_id")
            )
        ).alias("ranking")
    )
    .withColumn(
        "assinatura_top5",
        F.concat_ws(
            " | ",
            F.expr(
                "transform(ranking, x -> cast(x.destino_id as string))"
            )
        )
    )
)

# 2. Quantidade de rankings diferentes
resumo_rankings = rankings_usuarios.agg(
    F.count("*").alias("viajantes"),
    F.countDistinct("assinatura_top5").alias(
        "rankings_diferentes"
    )
)

display(resumo_rankings)

# 3. Rankings mais frequentes
display(
    rankings_usuarios
    .groupBy("assinatura_top5")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

# 4. Destinos mais presentes no Top 5
display(
    top5_rf
    .groupBy("destino_id", "destino")
    .agg(
        F.count("*").alias("vezes_top5"),
        F.avg("target").alias("taxa_sim")
    )
    .orderBy(F.desc("vezes_top5"))
    .limit(15)
)

print("Análise de diversidade concluída!")


In [0]:

# ================================================
# ETAPA 13 - TOP 5 SOBRE DESTINOS AVALIADOS
# RANDOM FOREST
# ================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.functions import vector_to_array

# Reaplicar o modelo mantendo identificação
# dos viajantes e destinos
previsoes_rf_ranking = modelo_rf.transform(
    teste_final.select(
        "usuario_id",
        "destino_id",
        "destino",
        "target",
        "features"
    ).withColumn(
        "label",
        F.col("target").cast("double")
    )
)

# Extrair a probabilidade de Sim
previsoes_rf_ranking = (
    previsoes_rf_ranking
    .withColumn(
        "prob_sim",
        vector_to_array("probability")[1]
    )
)

# Ranking por viajante
janela_ranking = Window.partitionBy(
    "usuario_id"
).orderBy(
    F.desc("prob_sim"),
    F.asc("destino_id")
)

ranking_rf = (
    previsoes_rf_ranking
    .withColumn(
        "posicao",
        F.row_number().over(janela_ranking)
    )
)

top5_rf = ranking_rf.filter(
    F.col("posicao") <= 5
)

# Quantos viajantes possuem avaliações suficientes?
display(
    previsoes_rf_ranking
    .groupBy("usuario_id")
    .count()
    .agg(
        F.count("*").alias("viajantes_teste"),
        F.sum(
            F.when(F.col("count") >= 5, 1).otherwise(0)
        ).alias("viajantes_com_5_ou_mais_avaliacoes")
    )
)

# Quantos positivos aparecem nas cinco posições?
display(
    top5_rf
    .groupBy("posicao")
    .agg(
        F.count("*").alias("recomendacoes"),
        F.sum("target").alias("avaliacoes_sim"),
        F.avg("target").alias("taxa_sim")
    )
    .orderBy("posicao")
)

print("Avaliação preliminar do Top 5 concluída!")


In [0]:

# ================================================
# ETAPA 12 - RANDOM FOREST
# STRAVL - MODELO 3
# ================================================

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F

# 1. Usar os conjuntos já preparados
treino_rf = treino_final.select(
    "features",
    F.col("target").cast("double").alias("label")
)

teste_rf = teste_final.select(
    "features",
    F.col("target").cast("double").alias("label")
)

# 2. Configuração inicial controlada
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=20,
    maxDepth=6,
    maxBins=32,
    minInstancesPerNode=100,
    featureSubsetStrategy="sqrt",
    seed=42
)

# 3. Treinar
print("Iniciando treinamento Random Forest...")

modelo_rf = rf.fit(treino_rf)

print("Random Forest treinado com sucesso!")

# 4. Aplicar ao teste
previsoes_rf = modelo_rf.transform(teste_rf)

# 5. Avaliar ROC-AUC
avaliador = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc_rf = avaliador.evaluate(previsoes_rf)

print("ROC-AUC Random Forest:", round(auc_rf, 4))

# 6. Matriz de confusão
display(
    previsoes_rf
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

# 7. Confirmar número de árvores
print("Quantidade de árvores:", len(modelo_rf.trees))


In [0]:

# ================================================
# STRAVL - ETAPA 11
# ÁRVORE DE DECISÃO - MODELO INICIAL
# ================================================

from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F

# 1. Preparar treino e teste
treino_arvore = treino_final.select(
    "features",
    F.col("target").cast("double").alias("label")
)

teste_arvore = teste_final.select(
    "features",
    F.col("target").cast("double").alias("label")
)

# 2. Configurar árvore com profundidade controlada
arvore = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=6,
    maxBins=32,
    minInstancesPerNode=100,
    seed=42
)

# 3. Treinar
print("Iniciando treinamento da Árvore de Decisão...")

modelo_arvore = arvore.fit(treino_arvore)

print("Árvore de Decisão treinada com sucesso!")

# 4. Previsões no conjunto de teste
previsoes_arvore = modelo_arvore.transform(teste_arvore)

# 5. Avaliação
avaliador = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc_arvore = avaliador.evaluate(previsoes_arvore)

print("ROC-AUC Árvore de Decisão:", round(auc_arvore, 4))

# 6. Matriz de confusão
display(
    previsoes_arvore
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

# 7. Profundidade e quantidade de nós
print("Profundidade:", modelo_arvore.depth)
print("Quantidade de nós:", modelo_arvore.numNodes)


In [0]:

# ETAPA 9 - VALIDAÇÃO SIMPLIFICADA

print("Validando vetor final...")

resultado = (
    amostra_final
    .select("features_com_interacoes")
    .first()
)

if resultado is not None:
    dimensoes = resultado[0].size

    print("Dimensões finais:", dimensoes)

    assert dimensoes == 25309

    print("SUCESSO! Vetor final validado.")
else:
    print("Nenhum registro encontrado.")


In [0]:

# ================================================
# ETAPA 9 - CORREÇÃO DO VETOR FINAL
# ================================================

from pyspark.ml.feature import VectorAssembler

# 1. Adicionar as features originais
# ao DataFrame que contém as interações

treino_interacoes_completo = (
    assembler.transform(treino_interacoes)
)

teste_interacoes_completo = (
    assembler.transform(teste_interacoes)
)

# 2. Combinar perfil, destino e interações
assembler_interacoes = VectorAssembler(
    inputCols=[
        "features",
        "vetor_interacoes"
    ],
    outputCol="features_com_interacoes"
)

# 3. Testar primeiro com 100 registros
amostra_final = assembler_interacoes.transform(
    treino_interacoes_completo.limit(100)
)

# 4. Validar o resultado
print("Validando vetor final...")

dimensoes = (
    amostra_final
    .select("features_com_interacoes")
    .first()[0]
    .size
)

print("Dimensões finais:", dimensoes)

assert dimensoes == 25309

print("Vetor final validado com sucesso!")

# 5. Visualizar exemplos
display(
    amostra_final.select(
        "usuario_id",
        "destino_id",
        "target",
        "features_com_interacoes"
    ).limit(5)
)


In [0]:

# ================================================
# ETAPA 8 - INTERAÇÕES PERFIL X DESTINO
# ================================================

from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

# Selecionar orçamento, estação e experiências
features_interacao = [
    c for c in colunas_features
    if c.startswith(("form_b_", "form_c_", "form_f_"))
]

print(
    "Características para interação:",
    len(features_interacao)
)

# Cada preferência será combinada com o destino.
# O vetor será esparso para economizar memória.

quantidade_destinos = destinos_indexados.count()
quantidade_interacoes = (
    len(features_interacao) * quantidade_destinos
)

print("Destinos:", quantidade_destinos)
print("Dimensões de interação:", quantidade_interacoes)

@udf(returnType=VectorUDT())
def criar_vetor_interacao(destino_index, *valores):
    if destino_index is None:
        return Vectors.sparse(
            quantidade_interacoes, [], []
        )

    indice_destino = int(destino_index)

    indices = []
    valores_ativos = []

    for i, valor in enumerate(valores):
        if valor is not None and float(valor) != 0:
            indice = (
                i * quantidade_destinos
                + indice_destino
            )
            indices.append(indice)
            valores_ativos.append(float(valor))

    return Vectors.sparse(
        quantidade_interacoes,
        indices,
        valores_ativos
    )

# Aplicar às bases de treino e teste
treino_interacoes = treino_ml.withColumn(
    "vetor_interacoes",
    criar_vetor_interacao(
        F.col("destino_index"),
        *[F.col(c) for c in features_interacao]
    )
)

teste_interacoes = teste_ml.withColumn(
    "vetor_interacoes",
    criar_vetor_interacao(
        F.col("destino_index"),
        *[F.col(c) for c in features_interacao]
    )
)

print("Preparação das interações concluída!")

display(
    treino_interacoes.select(
        "usuario_id",
        "destino_id",
        "target",
        "vetor_interacoes"
    ).limit(5)
)


In [0]:

# ETAPA 7 - ANALISAR O RANKING DO MODELO ATUAL

from pyspark.sql import functions as F

# Os 47 primeiros coeficientes são do perfil.
# Os demais representam os 1.486 destinos.
coeficientes = modelo_lr_balanceado.coefficients.toArray()

pesos_destinos = [
    (indice, float(peso))
    for indice, peso in enumerate(coeficientes[47:])
]

df_pesos = spark.createDataFrame(
    pesos_destinos,
    ["destino_index", "peso_modelo"]
)

ranking_destinos = (
    destinos_indexados
    .withColumn(
        "destino_index",
        F.col("destino_index").cast("long")
    )
    .join(
        df_pesos.withColumn(
            "destino_index",
            F.col("destino_index").cast("long")
        ),
        "destino_index"
    )
    .join(
        spark.table("workspace.bronze.destinos"),
        "destino_id"
    )
    .orderBy(F.desc("peso_modelo"))
)

print("Top 10 destinos favorecidos pelo modelo atual:")
display(
    ranking_destinos.select(
        "destino_id",
        "destino",
        "peso_modelo"
    ).limit(10)
)


In [0]:

# ================================================
# ETAPA 6 - TESTE DE PERSONALIZAÇÃO
# ================================================

from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

# Extrair probabilidade da classe Sim
previsoes_prob = (
    previsoes_balanceadas
    .withColumn(
        "prob_sim",
        vector_to_array("probability")[1]
    )
)

# Verificar variação das probabilidades
# entre viajantes para o mesmo destino
variacao_destinos = (
    previsoes_prob
    .groupBy("destino_id")
    .agg(
        F.countDistinct("usuario_id").alias("viajantes"),
        F.min("prob_sim").alias("prob_min"),
        F.max("prob_sim").alias("prob_max"),
        F.stddev("prob_sim").alias("desvio_prob")
    )
    .filter(F.col("viajantes") >= 10)
    .withColumn(
        "amplitude",
        F.col("prob_max") - F.col("prob_min")
    )
    .orderBy(F.desc("viajantes"))
)

display(variacao_destinos.limit(15))

# Resumo das amplitudes
display(
    variacao_destinos.agg(
        F.avg("amplitude").alias("amplitude_media"),
        F.avg("desvio_prob").alias("desvio_medio")
    )
)


In [0]:

# ================================================
# ETAPA 5 - REGRESSÃO LOGÍSTICA BALANCEADA
# ================================================

from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Calcular a distribuição somente do treino
contagens = {
    int(row["target"]): row["count"]
    for row in treino_final.groupBy("target").count().collect()
}

total = sum(contagens.values())

peso_nao = total / (2 * contagens[0])
peso_sim = total / (2 * contagens[1])

print("Peso Não:", round(peso_nao, 4))
print("Peso Sim:", round(peso_sim, 4))

# 2. Adicionar pesos ao conjunto de treino
treino_balanceado = treino_final.withColumn(
    "peso_classe",
    F.when(F.col("target") == 1, peso_sim)
     .otherwise(peso_nao)
)

# 3. Treinar o modelo com pesos
lr_balanceada = LogisticRegression(
    featuresCol="features",
    labelCol="target",
    weightCol="peso_classe",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0,
    standardization=False
)

print("Treinando modelo balanceado...")

modelo_lr_balanceado = lr_balanceada.fit(
    treino_balanceado.select(
        "features",
        "target",
        "peso_classe"
    )
)

print("Modelo balanceado treinado!")

# 4. Avaliar com o mesmo teste original
previsoes_balanceadas = modelo_lr_balanceado.transform(
    teste_final.select(
        "usuario_id",
        "destino_id",
        "target",
        "features"
    )
)

# 5. ROC-AUC
avaliador_auc = BinaryClassificationEvaluator(
    labelCol="target",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc_balanceado = avaliador_auc.evaluate(
    previsoes_balanceadas
)

print("ROC-AUC balanceado:", round(auc_balanceado, 4))

# 6. Matriz de confusão
display(
    previsoes_balanceadas
    .groupBy("target", "prediction")
    .count()
    .orderBy("target", "prediction")
)

# 7. Métricas específicas da classe Sim
matriz = {
    (int(r["target"]), int(r["prediction"])): r["count"]
    for r in previsoes_balanceadas
        .groupBy("target", "prediction")
        .count()
        .collect()
}

tp = matriz.get((1, 1), 0)
fp = matriz.get((0, 1), 0)
fn = matriz.get((1, 0), 0)
tn = matriz.get((0, 0), 0)

precision_sim = tp / (tp + fp) if tp + fp else 0
recall_sim = tp / (tp + fn) if tp + fn else 0

f1_sim = (
    2 * precision_sim * recall_sim /
    (precision_sim + recall_sim)
    if precision_sim + recall_sim else 0
)

print("\nMÉTRICAS DA CLASSE SIM")
print("=====================")
print(f"Precision Sim: {precision_sim:.4f}")
print(f"Recall Sim:    {recall_sim:.4f}")
print(f"F1 Sim:        {f1_sim:.4f}")


In [0]:

# ================================================
# ETAPA 4 - TREINAMENTO DO MODELO
# REGRESSÃO LOGÍSTICA - STRAVL
# ================================================

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.sql import functions as F

# 1. Configurar o modelo
lr = LogisticRegression(
    featuresCol="features",
    labelCol="target",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=30,
    regParam=0.1,
    elasticNetParam=0.0,
    standardization=False
)

# 2. Treinar somente com a base de treinamento
print("Iniciando treinamento...")

modelo_lr = lr.fit(
    treino_final.select("features", "target")
)

print("Modelo treinado com sucesso!")

# 3. Aplicar o modelo ao conjunto de teste
previsoes = modelo_lr.transform(
    teste_final.select(
        "usuario_id",
        "destino_id",
        "target",
        "features"
    )
)

# 4. Calcular ROC-AUC
avaliador_auc = BinaryClassificationEvaluator(
    labelCol="target",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = avaliador_auc.evaluate(previsoes)

# 5. Calcular Precision, Recall e F1
avaliador = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction"
)

precision = avaliador.evaluate(
    previsoes,
    {avaliador.metricName: "weightedPrecision"}
)

recall = avaliador.evaluate(
    previsoes,
    {avaliador.metricName: "weightedRecall"}
)

f1 = avaliador.evaluate(
    previsoes,
    {avaliador.metricName: "f1"}
)

# 6. Exibir métricas
print("\nRESULTADOS DO MODELO STRAVL")
print("==========================")
print(f"ROC-AUC:   {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

# 7. Exibir matriz de confusão
display(
    previsoes
    .groupBy("target", "prediction")
    .count()
    .orderBy("target", "prediction")
)

# 8. Exibir exemplos de previsões
display(
    previsoes.select(
        "usuario_id",
        "destino_id",
        "target",
        "prediction",
        "probability"
    ).limit(10)
)


In [0]:

# ================================================
# ETAPA 3 - PREPARAÇÃO DAS FEATURES DO MODELO
# ================================================

from pyspark.sql import functions as F
from pyspark.ml.feature import OneHotEncoder, VectorAssembler

# Identificar todos os destinos do catálogo
destinos_catalogo = (
    spark.table("workspace.bronze.destinos")
    .select("destino_id")
    .distinct()
)

# Criar mapeamento fixo de destinos
from pyspark.sql.window import Window

janela = Window.orderBy("destino_id")

destinos_indexados = (
    destinos_catalogo
    .withColumn(
        "destino_index",
        (F.row_number().over(janela) - 1).cast("double")
    )
)

# Aplicar o mesmo mapeamento ao treino e teste
treino_ml = treino_features.join(
    destinos_indexados,
    on="destino_id",
    how="inner"
)

teste_ml = teste_features.join(
    destinos_indexados,
    on="destino_id",
    how="inner"
)

# Codificação categórica dos destinos
encoder = OneHotEncoder(
    inputCols=["destino_index"],
    outputCols=["destino_vector"],
    dropLast=False
)

encoder_model = encoder.fit(treino_ml)

treino_ml = encoder_model.transform(treino_ml)
teste_ml = encoder_model.transform(teste_ml)

# Reunir as características de perfil e destino
assembler = VectorAssembler(
    inputCols=colunas_features + ["destino_vector"],
    outputCol="features",
    handleInvalid="error"
)

treino_final = assembler.transform(treino_ml)
teste_final = assembler.transform(teste_ml)

# Conferir resultado
print("Treino:", treino_final.count())
print("Teste:", teste_final.count())

print(
    "Dimensões do vetor:",
    treino_final.select("features").first()[0].size
)

display(
    treino_final.select(
        "usuario_id",
        "destino_id",
        "target",
        "features"
    ).limit(5)
)


In [0]:

# ================================================
# ETAPA 2 - ENGENHARIA DE FEATURES
# ================================================

from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler

# Mapeamento das perguntas e seus códigos válidos
perguntas = {
    "form_a": ["0", "1", "2", "3"],
    "form_b": ["0", "1", "2", "3"],
    "form_c": ["0", "1", "2", "3"],
    "form_f": [str(i) for i in range(8)],
    "form_g": [str(i) for i in range(8)],
    "form_h": ["0", "1", "2"],
    "form_i": ["0", "1", "2"],
    "form_j": ["0", "1", "2"],
    "form_r": ["0", "1"],
    "form_rr": ["e", "n", "c", "a", "s", "m", "f", "o"]
}

# Criar indicadores binários para cada resposta
def criar_features(df):
    resultado = df

    for pergunta, codigos in perguntas.items():
        coluna = f"{pergunta}_codigos"

        for codigo in codigos:
            nome_feature = f"{pergunta}_{codigo}"

            resultado = resultado.withColumn(
                nome_feature,
                F.when(
                    F.array_contains(
                        F.col(coluna).cast("array<string>"),
                        codigo
                    ),
                    F.lit(1.0)
                ).otherwise(F.lit(0.0))
            )

    # Codificar o destino como variável categórica.
    # O índice será preparado na próxima etapa.
    return resultado

treino_features = criar_features(treino)
teste_features = criar_features(teste)

# Listar características criadas
colunas_features = [
    f"{pergunta}_{codigo}"
    for pergunta, codigos in perguntas.items()
    for codigo in codigos
]

print("Total de características de perfil:", len(colunas_features))

# Exibir algumas características
display(
    treino_features.select(
        "usuario_id",
        "destino_id",
        "target",
        *colunas_features[:12]
    ).limit(10)
)

# Validar quantidade de características
assert len(colunas_features) == 47

print("Engenharia de features concluída!")


In [0]:

from pyspark.sql import functions as F

# ================================================
# STRAVL - MACHINE LEARNING
# ETAPA 1: PREPARAÇÃO DOS DADOS
# ================================================

# 1. Carregar Gold auditada
gold = spark.table(
    "workspace.gold.amostra_wide"
)

# 2. Selecionar avaliações binárias
# Sim = 1 | Não = 0
ml_base = (
    gold
    .filter(F.col("target").isin(0, 1))
    .select(
        "usuario_id",
        "destino_id",
        "destino",
        "target",
        "form_a_codigos",
        "form_b_codigos",
        "form_c_codigos",
        "form_f_codigos",
        "form_g_codigos",
        "form_h_codigos",
        "form_i_codigos",
        "form_j_codigos",
        "form_r_codigos",
        "form_rr_codigos",
        "orcamento_todas_faixas",
        "popularidade_todas_opcoes",
        "regiao_inconsistente"
    )
)

print("Total de avaliações para ML:", ml_base.count())
print("Total de variáveis:", len(ml_base.columns))

# 3. Separar viajantes, não avaliações
usuarios = (
    ml_base
    .select("usuario_id")
    .distinct()
)

usuarios_treino, usuarios_teste = (
    usuarios.randomSplit([0.8, 0.2], seed=42)
)

# 4. Criar bases de treino e teste
treino = ml_base.join(
    usuarios_treino,
    on="usuario_id",
    how="inner"
)

teste = ml_base.join(
    usuarios_teste,
    on="usuario_id",
    how="inner"
)

print("Avaliações de treino:", treino.count())
print("Avaliações de teste:", teste.count())

# 5. Validar que nenhum viajante
# aparece simultaneamente nos dois conjuntos
usuarios_repetidos = (
    usuarios_treino
    .join(
        usuarios_teste,
        on="usuario_id",
        how="inner"
    )
    .count()
)

print(
    "Viajantes presentes nos dois conjuntos:",
    usuarios_repetidos
)

assert usuarios_repetidos == 0

# 6. Distribuição do alvo
display(
    ml_base
    .groupBy("target")
    .count()
    .orderBy("target")
)
